# Cajeta - STA / Signoff (interactivo)

Capsula 8. El STA **no simula**: calcula, para cada camino, cuanto tarda la
senal (**arrival**) y cuanto tiempo tiene (**required**). El margen es el
**slack**. Aqui jugamos con el **reloj** del `semaforo` y vemos el slack
cerrar o violar -- con los numeros reales de la firma (OpenSTA / PrimeTime).

Corre las celdas en orden.

## 1 - El camino critico (datos reales de la firma)

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Camino critico de SETUP (del report_timing, capsula Tres Leches -> Cajeta):
#   FF _46_ (clk->Q) -> NOR2 -> salida 'red/yellow'. Incrementos en ns.
etapas = [
    ('clk -> clkbuf_0 (CTS)',        0.11),
    ('clkbuf_0 -> clkbuf_1 (CTS)',   0.15),
    ('FF _46_  CLK->Q',              0.54),
    ('NOR2 (logica)',                0.04),
]
ARRIVAL   = round(sum(d for _, d in etapas), 2)   # tiempo de llegada del dato
OUT_DELAY = 2.0     # output_delay del SDC (margen para el mundo externo)
HOLD_SLACK = 0.48   # el hold NO depende del periodo (es fijo)
print(f'Arrival (llegada del dato) = {ARRIVAL} ns')
print(f'Output delay (SDC)         = {OUT_DELAY} ns')
print(f'Hold slack (fijo)          = +{HOLD_SLACK} ns')

## 2 - ¿A donde se va el tiempo?

El arrival es la suma de retardos a lo largo del camino: el reloj llegando por
los buffers del CTS, el flip-flop disparando, y la logica. Esto es lo que el
STA calcula para **cada** camino del chip.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
acc = 0.0
for nombre, d in etapas:
    ax.barh(0, d, left=acc, edgecolor='white')
    if d >= 0.05:
        ax.text(acc + d/2, 0, f'{nombre}\n{d:.2f}', ha='center', va='center', fontsize=7)
    acc += d
ax.set_xlim(0, ARRIVAL*1.1); ax.set_yticks([])
ax.set_xlabel('ns (acumulado)')
ax.set_title(f'Camino critico: arrival = {ARRIVAL} ns')
plt.tight_layout(); plt.show()

## 3 - Juega con el reloj: ¿cierra o viola?

El **setup** exige que el dato llegue **antes** del siguiente tic:

> slack_setup = **periodo** - output_delay - arrival

Sube la frecuencia (baja el periodo) y mira cuando el slack se vuelve negativo:
ahi el chip **ya no cumple**. La frecuencia donde slack = 0 es la **maxima**.

In [ ]:
def firma(frecuencia_MHz=100.0):
    T = 1000.0 / frecuencia_MHz                 # periodo en ns
    disponible = T - OUT_DELAY                  # required time
    slack = disponible - ARRIVAL
    fmax = 1000.0 / (ARRIVAL + OUT_DELAY)        # donde slack = 0
    fig, ax = plt.subplots(figsize=(7, 3))
    col = '#16a34a' if slack >= 0 else '#dc2626'
    ax.barh(['disponible\n(T - out_delay)', 'llegada\n(arrival)'],
            [disponible, ARRIVAL], color=[col, '#C8742B'])
    ax.axvline(ARRIVAL, ls='--', color='#C8742B', lw=1)
    estado = 'CIERRA (MET)' if slack >= 0 else 'VIOLA (setup)'
    ax.set_title(f'f = {frecuencia_MHz:.0f} MHz  (T = {T:.2f} ns)   '
                 f'slack = {slack:+.2f} ns  ->  {estado}')
    ax.set_xlabel('ns')
    ax.text(0.98, 0.05, f'f_max ~ {fmax:.0f} MHz', transform=ax.transAxes,
            ha='right', fontsize=9, color='#555')
    plt.tight_layout(); plt.show()

interact(firma, frecuencia_MHz=FloatSlider(value=100, min=80, max=420, step=10,
         description='f (MHz)'));

## 4 - Setup vs Hold, y los dos STA

- **Setup** (lo de arriba): no llegar **tarde** -> limita la frecuencia maxima.
- **Hold**: no llegar **demasiado temprano** -> **no** depende del periodo; aqui
  el slack de hold es **+0.48 ns** (fijo), tambien cumple.

Mismo diseno, mismo SPEF, dos herramientas de signoff -- practicamente el mismo veredicto:

| | OpenSTA (OSS) | PrimeTime (Synopsys) |
|---|---|---|
| slack setup | +7.16 ns | +7.38 ns |
| slack hold  | +0.49 ns | +0.48 ns |
| violaciones | 0 | 0 |

Si el slack cierra en todos lados -> **timing cerrado**: el chip se puede fabricar.

## Lo que sigue

Cerramos el **camino digital**: del RTL al layout **firmado**. Pero un chip no es
solo logica -- falta el mundo **analogico**, donde no hay sintesis y todo se
dibuja a mano.

🍮 Cajeta &nbsp;->&nbsp; 🧅 Bunuelo (Esquematico)